In [3]:
import cv2
import numpy as np
import os

# Создание папки для сохранения кадров
output_folder = "saved_frames"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Загрузка модели YOLO
net = cv2.dnn.readNet("yolov4.weights", "yolov4.cfg")

# Получение имен всех слоев
layer_names = net.getLayerNames()

# Получение выходных слоев
output_layers = [layer_names[i - 1] for i in net.getUnconnectedOutLayers()]

# Загрузка классов
with open("coco.names", "r") as f:
    classes = [line.strip() for line in f.readlines()]

# Захват видеопотока
cap = cv2.VideoCapture("video_2025-01-19_21-33-09.mp4")

# Определение области интереса (ROI)
roi = [(130, 130), (600, 600)]  # Пример координат области

# Счетчик кадров
frame_count = 0

# Размер батча (количество кадров, обрабатываемых одновременно)
batch_size = 4

# Список для хранения кадров и их метаданных
frames = []
metadata = []

# Счетчик для пропуска каждого второго кадра
skip_counter = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Пропуск каждого второго кадра
    skip_counter += 1
    if skip_counter % 2 != 0:
        continue

    # Уменьшение разрешения кадра на 10%
    height, width = frame.shape[:2]
    new_width = int(width * 0.9)
    new_height = int(height * 0.9)
    frame = cv2.resize(frame, (new_width, new_height))

    # Добавление кадра в батч
    frames.append(frame)
    metadata.append((new_height, new_width))

    # Если батч заполнен, обрабатываем его
    if len(frames) == batch_size:
        # Подготовка батча для YOLO
        blob = cv2.dnn.blobFromImages(frames, 0.00392, (416, 416), (0, 0, 0), True, crop=False)
        net.setInput(blob)
        outs = net.forward(output_layers)

        # Обработка результатов детекции для каждого кадра в батче
        for i in range(len(frames)):
            height, width = metadata[i]
            frame = frames[i]

            # Обработка результатов детекции
            class_ids = []
            confidences = []
            boxes = []
            for out in outs:
                for detection in out[i]:  # Обрабатываем результаты для i-го кадра
                    scores = detection[5:]
                    class_id = np.argmax(scores)
                    confidence = scores[class_id]
                    if confidence > 0.5 and classes[class_id] == "car":  # Проверка класса "car"
                        center_x = int(detection[0] * width)
                        center_y = int(detection[1] * height)
                        w = int(detection[2] * width)
                        h = int(detection[3] * height)

                        x = int(center_x - w / 2)
                        y = int(center_y - h / 2)

                        boxes.append([x, y, w, h])
                        confidences.append(float(confidence))
                        class_ids.append(class_id)

            # Применение Non-Maximum Suppression для удаления дубликатов
            indexes = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, 0.4)

            # Проверка, что bounding box машины пересекается с ROI или полностью внутри
            for j in range(len(boxes)):
                if j in indexes:
                    x, y, w, h = boxes[j]

                    # Проверка пересечения bounding box и ROI
                    x1 = max(x, roi[0][0])
                    y1 = max(y, roi[0][1])
                    x2 = min(x + w, roi[1][0])
                    y2 = min(y + h, roi[1][1])

                    # Если есть пересечение или bounding box полностью внутри ROI
                    if (x2 > x1 and y2 > y1) or (x >= roi[0][0] and y >= roi[0][1] and
                                                 x + w <= roi[1][0] and y + h <= roi[1][1]):
                        # Увеличение счетчика кадров
                        frame_count += 1

                        # Обрезка кадра по ROI
                        roi_frame = frame[roi[0][1]:roi[1][1], roi[0][0]:roi[1][0]]

                        # Сохранение обрезанного кадра
                        output_path = os.path.join(output_folder, f"frame_{frame_count}.jpg")
                        cv2.imwrite(output_path, roi_frame)
                        print(f"Сохранен кадр: {output_path}")

            # Отображение кадра с выделенными объектами и ROI
            cv2.rectangle(frame, roi[0], roi[1], (0, 255, 0), 2)
            for j in range(len(boxes)):
                if j in indexes:
                    x, y, w, h = boxes[j]
                    cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 0, 255), 2)
            cv2.imshow("Frame", frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        # Очистка батча
        frames = []
        metadata = []

cap.release()
cv2.destroyAllWindows()

Сохранен кадр: saved_frames\frame_1.jpg
Сохранен кадр: saved_frames\frame_2.jpg
Сохранен кадр: saved_frames\frame_3.jpg
Сохранен кадр: saved_frames\frame_4.jpg
Сохранен кадр: saved_frames\frame_5.jpg
Сохранен кадр: saved_frames\frame_6.jpg
Сохранен кадр: saved_frames\frame_7.jpg
Сохранен кадр: saved_frames\frame_8.jpg
Сохранен кадр: saved_frames\frame_9.jpg
Сохранен кадр: saved_frames\frame_10.jpg
Сохранен кадр: saved_frames\frame_11.jpg
Сохранен кадр: saved_frames\frame_12.jpg
Сохранен кадр: saved_frames\frame_13.jpg
Сохранен кадр: saved_frames\frame_14.jpg
Сохранен кадр: saved_frames\frame_15.jpg
Сохранен кадр: saved_frames\frame_16.jpg
Сохранен кадр: saved_frames\frame_17.jpg
Сохранен кадр: saved_frames\frame_18.jpg
Сохранен кадр: saved_frames\frame_19.jpg
Сохранен кадр: saved_frames\frame_20.jpg
Сохранен кадр: saved_frames\frame_21.jpg
Сохранен кадр: saved_frames\frame_22.jpg
Сохранен кадр: saved_frames\frame_23.jpg
Сохранен кадр: saved_frames\frame_24.jpg
Сохранен кадр: saved_fram